# Fabric Private VNet — Smoke Test

Validates a deployed `fabric-private-terraform` stack from any workstation that has:

- `az login` to the target subscription (MCAPS: use cached `az` token, **not** SP — Conditional Access blocks SP data-plane).
- Python 3.10+ with `azure-identity`, `azure-mgmt-resource`, `azure-mgmt-network`, `azure-mgmt-fabric`, `dnspython`, `python-hcl2`.
- Network path to the resources (run from inside the hub VNet, the simulated on-prem subnet, or a peered ExpressRoute / VPN edge — anywhere DNS-Resolver-inbound is reachable).

It loads CAF-spec names from `terraform output caf_naming -json`, queries Azure to verify each component, and prints a green/red checklist.

## 0. Install dependencies (one-time)

In [ ]:
%pip install -q azure-identity azure-mgmt-resource azure-mgmt-network azure-mgmt-fabric dnspython requests

## 1. Load CAF naming + Terraform outputs

Reads `terraform output -json` from the parent directory. Falls back to env vars if Terraform state isn't local.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

TF_DIR = Path(os.environ.get('FABRIC_TF_DIR', Path.cwd().parent)).resolve()
print(f'Terraform working dir: {TF_DIR}')

def tf_output():
    try:
        r = subprocess.run(['terraform', 'output', '-json'], cwd=TF_DIR, capture_output=True, text=True, check=True)
        return json.loads(r.stdout)
    except (subprocess.CalledProcessError, FileNotFoundError) as e:
        print(f'terraform output failed: {e}')
        return {}

tf = tf_output()
if not tf:
    raise RuntimeError('No terraform outputs. cd to fabric-private-terraform/ and `terraform apply` first, or set FABRIC_TF_DIR.')

caf = tf['caf_naming']['value']
rg_name = tf['resource_group_name']['value']
fabric_capacity_name = tf['fabric_capacity_name']['value']
fabric_capacity_id = tf['fabric_capacity_id']['value']
private_endpoint_ip = tf['private_endpoint_ip']['value']
dns_resolver_inbound_ip = tf['dns_resolver_inbound_ip']['value']
private_dns_zones = caf['private_dns_zones']

print(f'Resource group   : {rg_name}')
print(f'CAF prefix       : {caf["suffix"]}  (instance={caf["suffix"].split("-")[-1]})')
print(f'Fabric capacity  : {fabric_capacity_name}')
print(f'Fabric PE IP     : {private_endpoint_ip}')
print(f'DNS Resolver IP  : {dns_resolver_inbound_ip}')

## 2. Auth + subscription discovery

Use `AzureCliCredential` per MCAPS tenant Conditional Access rules (SP data-plane is denied).

In [ ]:
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient, SubscriptionClient
from azure.mgmt.network import NetworkManagementClient

cred = AzureCliCredential()
sub_client = SubscriptionClient(cred)
sub_id = next(iter(sub_client.subscriptions.list())).subscription_id
print(f'Subscription: {sub_id}')

rm = ResourceManagementClient(cred, sub_id)
net = NetworkManagementClient(cred, sub_id)

## 3. Helper — pass/fail checklist

In [ ]:
results = []
def check(name, ok, detail=''):
    icon = '[OK]  ' if ok else '[FAIL]'
    line = f'{icon} {name:<60} {detail}'
    print(line)
    results.append((name, ok, detail))
    return ok

## 4. RG + tags

In [ ]:
try:
    rg = rm.resource_groups.get(rg_name)
    check('Resource group exists', True, rg.location)
    check('RG has managed_by=terraform tag', (rg.tags or {}).get('managed_by') == 'terraform', str(rg.tags))
    check('RG matches CAF name', rg_name == caf['rg'], f'actual={rg_name} caf={caf["rg"]}')
except Exception as e:
    check('Resource group exists', False, str(e))

## 5. VNet + subnets

In [ ]:
from azure.core.exceptions import ResourceNotFoundError
expected_vnets = [caf['vnet']]
for v in expected_vnets:
    # Vnet may still carry pre-CAF name if rename hasn't been applied -- discover by tag.
    found = [x for x in net.virtual_networks.list(rg_name) if x.name == v or 'fabric' in (x.tags or {}).get('workload', '').lower()]
    if not found:
        check(f'VNet {v}', False, 'not found')
        continue
    vn = found[0]
    check(f'VNet {vn.name}', True, vn.address_space.address_prefixes[0])
    expected_subnets = {'AzureFirewallSubnet', 'snet-dnsr-inbound', 'snet-dnsr-outbound', 'snet-private-endpoints', 'snet-jumpbox',
                       caf['snet_dnsr_in'], caf['snet_dnsr_out'], caf['snet_pe'], caf['snet_jumpbox']}
    present = {s.name for s in vn.subnets}
    matched = expected_subnets & present
    check('  >= 4 expected subnets present', len(matched) >= 4, sorted(matched))

## 6. Private DNS zones + VNet links

In [ ]:
from azure.mgmt.privatedns import PrivateDnsManagementClient
pdns = PrivateDnsManagementClient(cred, sub_id)

for z in private_dns_zones:
    try:
        zone = pdns.private_zones.get(rg_name, z)
        check(f'Private DNS zone {z}', True, f'records={zone.number_of_record_sets}')
        links = list(pdns.virtual_network_links.list(rg_name, z))
        check(f'  zone {z} has >= 1 VNet link', len(links) >= 1, f'{len(links)} link(s)')
        a_records = [r for r in pdns.record_sets.list_by_type(rg_name, z, 'A')]
        check(f'  zone {z} has A records', len(a_records) >= 1, f'{len(a_records)} A record(s)')
    except ResourceNotFoundError:
        check(f'Private DNS zone {z}', False, 'not found')

## 7. DNS Resolver inbound endpoint

In [ ]:
from azure.mgmt.dnsresolver import DnsResolverManagementClient
dr = DnsResolverManagementClient(cred, sub_id)

resolvers = list(dr.dns_resolvers.list_by_resource_group(rg_name))
if not resolvers:
    check('Private DNS Resolver', False, 'none in RG')
else:
    rv = resolvers[0]
    check(f'Private DNS Resolver {rv.name}', rv.provisioning_state == 'Succeeded', rv.provisioning_state)
    inb = list(dr.inbound_endpoints.list(rg_name, rv.name))
    if inb:
        ip = inb[0].ip_configurations[0].private_ip_address
        check(f'  Inbound endpoint IP', ip == dns_resolver_inbound_ip, f'{ip} (TF says {dns_resolver_inbound_ip})')
    else:
        check('  Inbound endpoint', False, 'none')

## 8. Private endpoint to Fabric — state + DNS

In [ ]:
pes = list(net.private_endpoints.list(rg_name))
fabric_pe = next((p for p in pes if 'fabric' in p.name.lower()), None)
if fabric_pe is None:
    check('Fabric private endpoint exists', False, 'none')
else:
    check(f'Fabric PE {fabric_pe.name}', True, fabric_pe.provisioning_state)
    conns = fabric_pe.private_link_service_connections or fabric_pe.manual_private_link_service_connections
    state = conns[0].private_link_service_connection_state.status if conns else 'Unknown'
    check('  PE connection state == Approved', state == 'Approved', state)
    nic = net.network_interfaces.get(rg_name, fabric_pe.network_interfaces[0].id.split('/')[-1])
    pe_ip = nic.ip_configurations[0].private_ip_address
    check('  PE NIC IP matches TF output', pe_ip == private_endpoint_ip, f'nic={pe_ip} tf={private_endpoint_ip}')
    check('  PE IP is in 10.50.3.0/24 (PE subnet)', pe_ip.startswith('10.50.3.'), pe_ip)

## 9. Fabric capacity — Active + correct SKU

In [ ]:
try:
    from azure.mgmt.fabric import FabricMgmtClient
except ImportError:
    print('azure-mgmt-fabric not installed; using azure-mgmt-resource generic get')
    FabricMgmtClient = None

if FabricMgmtClient:
    fab = FabricMgmtClient(cred, sub_id)
    cap = fab.fabric_capacities.get(rg_name, fabric_capacity_name)
    check(f'Fabric capacity {cap.name}', cap.properties.state == 'Active', cap.properties.state)
    check('  SKU = F2 (lowest paid)', cap.sku.name == 'F2', f'{cap.sku.name} / {cap.sku.tier}')
    check('  admin members configured', len(cap.properties.administration.members) >= 1, f'{len(cap.properties.administration.members)} member(s)')
else:
    cap = rm.resources.get_by_id(fabric_capacity_id, api_version='2023-11-01')
    state = cap.properties.get('state')
    check(f'Fabric capacity {cap.name}', state == 'Active', state)

## 10. DNS resolution — Fabric FQDNs resolve to private IPs (10.50.x.x)

This proves the **end-to-end private path**: client → DNS Resolver inbound → private DNS zone A-record → 10.x.x.x. If you see public IPs (52.x, 20.x) the PE isn't wired correctly.

In [ ]:
import socket
import dns.resolver

fabric_fqdns = tf.get('fabric_fqdns', {}).get('value', {})
if not fabric_fqdns:
    print('No fabric_fqdns output; using fallback test FQDNs')
    fabric_fqdns = {'api': 'api.fabric.microsoft.com'}

# Force resolution through the deployed DNS Resolver inbound IP when reachable.
resolver = dns.resolver.Resolver()
try:
    socket.create_connection((dns_resolver_inbound_ip, 53), timeout=2).close()
    resolver.nameservers = [dns_resolver_inbound_ip]
    print(f'Using DNS Resolver inbound IP {dns_resolver_inbound_ip} for resolution')
except OSError:
    print(f'DNS Resolver {dns_resolver_inbound_ip}:53 unreachable from here — falling back to system resolver. Run this notebook from inside the VNet / on-prem subnet for a true test.')

for label, fqdn in fabric_fqdns.items():
    try:
        ans = resolver.resolve(fqdn, 'A', lifetime=5)
        ips = [r.address for r in ans]
        private = all(ip.startswith(('10.', '172.', '192.168.')) for ip in ips)
        check(f'DNS {label}: {fqdn}', private, ', '.join(ips))
    except Exception as e:
        check(f'DNS {label}: {fqdn}', False, str(e))

## 11. Firewall policy — required Fabric rule collections present

In [ ]:
policies = list(net.firewall_policies.list(rg_name))
if not policies:
    check('Azure Firewall policy', False, 'none (enable_firewall=false?)')
else:
    fp = policies[0]
    check(f'Firewall policy {fp.name}', fp.provisioning_state == 'Succeeded', fp.provisioning_state)
    rcgs = list(net.firewall_policy_rule_collection_groups.list(rg_name, fp.name))
    expected = {'rcg-fabric-network', 'rcg-fabric-application'}
    present = {r.name for r in rcgs}
    check('  required rule collection groups', expected.issubset(present), sorted(present))

## 12. CAF naming compliance snapshot

Shows actual deployed names vs CAF-recommended names. Use this to decide whether to rename on next destroy/recreate cycle.

In [ ]:
actual = {}
for v in net.virtual_networks.list(rg_name):
    actual['vnet'] = v.name
    for s in v.subnets:
        actual[f'subnet:{s.name}'] = s.name
for p in net.private_endpoints.list(rg_name):
    actual[f'pe:{p.name}'] = p.name
for f in net.azure_firewalls.list(rg_name):
    actual['firewall'] = f.name
actual['rg'] = rg_name
actual['fabric_capacity'] = fabric_capacity_name

import pandas as pd
rows = []
for key, caf_name in [('rg', caf['rg']), ('vnet', caf['vnet']), ('snet_pe', caf['snet_pe']),
                       ('snet_dnsr_in', caf['snet_dnsr_in']), ('snet_dnsr_out', caf['snet_dnsr_out']),
                       ('snet_jumpbox', caf['snet_jumpbox']), ('fabric_capacity', caf['fabric_capacity']),
                       ('pep_fabric', caf['pep_fabric']), ('afw', caf['afw']), ('afwp', caf['afwp']),
                       ('dnspr', caf['dnspr'])]:
    a = actual.get(key) or actual.get(f'subnet:{caf_name}') or actual.get(f'pe:{caf_name}') or '(not found / pre-CAF name)'
    rows.append({'resource': key, 'caf_recommended': caf_name, 'actual_deployed': a, 'compliant': a == caf_name})
df = pd.DataFrame(rows)
df

## 13. Summary

In [ ]:
passed = sum(1 for _, ok, _ in results if ok)
total = len(results)
print(f'\n{"="*70}')
print(f'  SMOKE TEST: {passed}/{total} checks passed')
print(f'{"="*70}\n')
fails = [(n, d) for n, ok, d in results if not ok]
if fails:
    print('FAILURES:')
    for n, d in fails:
        print(f'  - {n}: {d}')
else:
    print('All green. Stack is healthy.')